In [53]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.linalg import solve_banded
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
np.random.seed(42)

In [ ]:
class BinaryOptionPricer:
    def __init__(self, S0, K, T, r, sigma):
        self.S0 = S0
        self.K = K
        self.T = T
        self.r = r
        self.sigma = sigma
    
    def analytical_solution(self, S, t):
        if t >= self.T:
            return np.where(S >= self.K, 1.0, 0.0)
        tau = self.T - t
        d2 = (np.log(S / self.K) + (self.r - 0.5 * self.sigma**2) * tau) / (self.sigma * np.sqrt(tau))
        
        return np.exp(-self.r * tau) * norm.cdf(d2)
    
    def implicit_scheme(self, N=100, M=200):

        dt = self.T / N
        S_max = 3 * self.S0
        dS = S_max / M
        S = np.linspace(0, S_max, M + 1)
        V = np.where(S >= self.K, 1.0, 0.0)
        for n in range(N - 1, -1, -1):
            V_new = np.zeros(M + 1)
            V_new[0] = 0.0  
            V_new[M] = np.exp(-self.r * (N - n) * dt)  

            ab = np.zeros((3, M - 1))
            rhs = np.zeros(M - 1)

            for i in range(1, M):
                Si = S[i]
                a = 0.5 * self.sigma**2 * Si**2 / dS**2
                b = self.r * Si / (2 * dS)
                ab[1, i-1] = 1 + dt * (2*a + self.r)
                if i < M - 1:
                    ab[0, i] = -dt * (a + b)
                if i > 1:
                    ab[2, i-2] = -dt * (a - b)

                if i == 1:
                    rhs[i-1] = V[i] + dt * (a - b) * V_new[0]
                elif i == M - 1:
                    rhs[i-1] = V[i] + dt * (a + b) * V_new[M]
                else:
                    rhs[i-1] = V[i]

            V_interior = solve_banded((1, 1), ab, rhs)
            V_new[1:M] = V_interior
            V = V_new.copy()
        
        return S, V
    
    def crank_nicolson_scheme(self, N=100, M=200):
        dt = self.T / N
        S_max = 3 * self.S0
        dS = S_max / M
        S = np.linspace(0, S_max, M + 1)
        V = np.where(S >= self.K, 1.0, 0.0)
        for n in range(N - 1, -1, -1):
            V_new = np.zeros(M + 1)

            V_new[0] = 0.0  
            V_new[M] = np.exp(-self.r * (N - n) * dt)  
            ab = np.zeros((3, M - 1))
            rhs = np.zeros(M - 1)

            for i in range(1, M):
                Si = S[i]

                a = 0.5 * self.sigma**2 * Si**2 / dS**2
                b = self.r * Si / (2 * dS)
                theta = 0.5
                ab[1, i-1] = 1 + theta * dt * (2*a + self.r)
                if i < M - 1:
                    ab[0, i] = -theta * dt * (a + b)
                if i > 1:
                    ab[2, i-2] = -theta * dt * (a - b)

                if i == 1: #Solving for the RHS 
                    rhs[i-1] = V[i] * (1 - (1-theta) * dt * (2*a + self.r))
                    rhs[i-1] += (1-theta) * dt * (a - b) * V[i-1]
                    rhs[i-1] += (1-theta) * dt * (a + b) * V[i+1]
                    rhs[i-1] += theta * dt * (a - b) * V_new[0]
                elif i == M - 1:
                    rhs[i-1] = V[i] * (1 - (1-theta) * dt * (2*a + self.r))
                    rhs[i-1] += (1-theta) * dt * (a - b) * V[i-1]
                    rhs[i-1] += (1-theta) * dt * (a + b) * V[i+1]
                    rhs[i-1] += theta * dt * (a + b) * V_new[M]
                else:
                    rhs[i-1] = V[i] * (1 - (1-theta) * dt * (2*a + self.r))
                    rhs[i-1] += (1-theta) * dt * (a - b) * V[i-1]
                    rhs[i-1] += (1-theta) * dt * (a + b) * V[i+1]
            
            V_interior = solve_banded((1, 1), ab, rhs)
            V_new[1:M] = V_interior
            V = V_new.copy()
        
        return S, V
    def option_surface_and_delta(self, S_min=0, S_max=None, t_steps=50, S_steps=200):

        if S_max is None:
            S_max = 3 * self.S0
        S_vals = np.linspace(S_min, S_max, S_steps)
        t_vals = np.linspace(0, self.T, t_steps)

        C = np.zeros((t_steps, S_steps))
        
        for i, t in enumerate(t_vals):
            C[i, :] = self.analytical_solution(S_vals, t)
   
        dS = S_vals[1] - S_vals[0] #Delta computation 
        delta = np.zeros(S_steps)

        delta[1:-1] = (C[0, 2:] - C[0, :-2]) / (2 * dS)
        delta[0] = (C[0, 1] - C[0, 0]) / dS
        delta[-1] = (C[0, -1] - C[0, -2]) / dS
        #Surface plotting
        fig = plt.figure(figsize=(16, 10))
        ax = fig.add_subplot(111, projection='3d')
        S_grid, t_grid = np.meshgrid(S_vals, t_vals)
        
        surf = ax.plot_surface(S_grid, t_grid, C, cmap='viridis', edgecolor='none')
        ax.set_xlabel('Stock Price S', fontsize=17)
        ax.set_ylabel('Time t', fontsize=17)
        ax.set_zlabel('Option Price c(S,t)', fontsize=17)
        ax.set_title('Digital Option Price Surface', fontsize=20)

        ax.tick_params(axis='x', labelsize=16, size=16)
        ax.tick_params(axis='y', labelsize=16, size=16)
        ax.tick_params(axis='z', labelsize=16, size=16)

        fig.colorbar(surf, shrink=0.5, aspect=10)
        plt.show()
        

        plt.figure(figsize=(10,6))
        plt.plot(S_vals, delta, label='Delta at t=0', color='green')
        plt.xlabel('Stock Price S', fontsize=16)
        plt.ylabel('Delta', fontsize=16)
        plt.title('Delta of Digital Option at t=0', fontsize=20)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=14)
        plt.show()
        
        return S_grid, t_grid, C, delta

In [32]:
S0 = 100    # Initial stock price
K = 105     # Strike price
T = 1.0     # Time to maturity
r = 0.05    # Risk-free rate
sigma = 0.2 # Volatility

In [33]:
pricer = BinaryOptionPricer(S0, K, T, r, sigma)

In [34]:
S_implicit, V_implicit = pricer.implicit_scheme(N=500, M=500)
S_cn, V_cn = pricer.crank_nicolson_scheme(N=500, M=500)

In [ ]:
V_analytical = pricer.analytical_solution(S_implicit, 0)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(S_implicit, V_analytical, 'b-', linewidth=2, label='Analytical')
ax.plot(S_implicit, V_implicit, 'r--', linewidth=2, label='Implicit FD')
ax.plot(S_cn, V_cn, 'g:', linewidth=2, label='Crank-Nicolson')
ax.set_xlabel('Stock Price (S)')
ax.set_ylabel('Option Value')
ax.set_title('Binary Call Option: Method Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(80, 120)
plt.tight_layout()
plt.show()

In [ ]:
idx_S0 = np.argmin(np.abs(S_implicit - S0))
print(f"\nNumerical Results at S = {S0}:")
print(f"Analytical solution: {V_analytical[idx_S0]:.6f}")
print(f"Implicit scheme:     {V_implicit[idx_S0]:.6f}")
print(f"Crank-Nicolson:      {V_cn[idx_S0]:.6f}")
print(f"Implicit error:      {abs(V_implicit[idx_S0] - V_analytical[idx_S0]):.2e}")
print(f"Crank-Nicolson error: {abs(V_cn[idx_S0] - V_analytical[idx_S0]):.2e}")

print(f"\nComparison around strike price K = {K}:")
for S_test in [95, 100, 105]:
    idx = np.argmin(np.abs(S_implicit - S_test))
    print(f"S = {S_test}:")
    print(f"  Analytical: {V_analytical[idx]:.6f}")
    print(f"  Implicit:   {V_implicit[idx]:.6f}")
    print(f"  C-N:        {V_cn[idx]:.6f}")

In [ ]:
pricer.option_surface_and_delta()

In [54]:
def delta_tau(z, tau, sigma, r, sign=1):
    if z <= 0:
        raise ValueError("z must be positive")
    numerator = np.log(z) + (r + sign * 0.5 * sigma**2) * tau
    denominator = sigma * np.sqrt(tau)
    return numerator / denominator

def barrier_call_price(S, K, B, T, t, r, sigma, M_t_less_B=True):
    tau = T - t  

    d_plus_S_K = delta_tau(S / K, tau, sigma, r, +1)
    d_plus_S_B = delta_tau(S / B, tau, sigma, r, +1)
    term1 = S * (norm.cdf(d_plus_S_K) - norm.cdf(d_plus_S_B))
    
    d_plus_B2_KS = delta_tau(B**2 / (K * S), tau, sigma, r, +1)
    d_plus_B_S = delta_tau(B / S, tau, sigma, r, +1)
    term2 = -S * (B / S)**(1 + 2 * (r / sigma**2)) * (norm.cdf(d_plus_B2_KS) - norm.cdf(d_plus_B_S))
    
    d_minus_S_K = delta_tau(S / K, tau, sigma, r, -1)
    d_minus_S_B = delta_tau(S / B, tau, sigma, r, -1)
    term3 = -K * np.exp(-r * tau) * (norm.cdf(d_minus_S_K) - norm.cdf(d_minus_S_B))
    
    d_minus_B2_KS = delta_tau(B**2 / (K * S), tau, sigma, r, -1)
    d_minus_B_S = delta_tau(B / S, tau, sigma, r, -1)
    term4 = (S / B)**(1 - (2 * r / sigma**2)) * np.exp(-r * tau) * K * (norm.cdf(d_minus_B2_KS) - norm.cdf(d_minus_B_S))
    
    return max(term1 + term2 + term3 + term4, 0.0)

def monte_carlo_barrier_call(S0, K, H, T, r, sigma, m, N_sim=100000):
    dt = T / m
    discount = np.exp(-r * T)
    payoffs = []
    
    for _ in range(N_sim):
        prices = [S0]
        barrier_breached = False
        
        for _ in range(m):
            Z = np.random.normal()
            S_new = prices[-1] * np.exp((r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z)
            if S_new >= H:
                barrier_breached = True
                break
            prices.append(S_new)
        
        if not barrier_breached:
            ST = prices[-1]
            payoff = max(ST - K, 0)
        else:
            payoff = 0
        payoffs.append(payoff)
    
    return discount * np.mean(payoffs)


In [43]:
beta1 = -0.5826  
S0 = 100
K = 90
H = 110
T = 1.0
r = 0.05
sigma = 0.2
ms = [10, 20, 40, 80, 160, 200, 250, 300, 500, 1000]
errors_raw = []
errors_corrected = []


In [ ]:
C_continuous = barrier_call_price(S0, K, H, T, 0, r, sigma)
print(f"Continuous barrier call price: {C_continuous:.6f}")

for m in ms:
    C_discrete = monte_carlo_barrier_call(S0, K, H, T, r, sigma, m, N_sim=50000)
    H_adj = H * np.exp(beta1 * sigma * np.sqrt(T / m))
    C_corrected = barrier_call_price(S0, K, H_adj, T, 0, r, sigma)
    error_raw = abs(C_discrete - C_continuous)
    error_corrected = abs(C_discrete - C_corrected)
    errors_raw.append(error_raw)
    errors_corrected.append(error_corrected)
    print(f"m={m}: MC={C_discrete:.4f}, Corrected barrier={H_adj:.4f}, C(H*)={C_corrected:.4f}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.loglog(ms, errors_raw, 'ro-', linewidth=2, markersize=8, label='Raw Error')
plt.loglog(ms, errors_corrected, 'bs-', linewidth=2, markersize=8, label='BGK Corrected Error')
plt.xlabel('Number of Monitoring Steps (m)')
plt.ylabel('Absolute Error')
plt.title('Convergence Analysis: Error vs m')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
inv_sqrt_m = [1/np.sqrt(m) for m in ms]
plt.loglog(inv_sqrt_m, errors_raw, 'ro-', linewidth=2, markersize=8, label='Raw Error')
plt.loglog(inv_sqrt_m, errors_corrected, 'bs-', linewidth=2, markersize=8, label='BGK Corrected Error')
plt.xlabel('1/√m', fontsize=18)
plt.ylabel('Absolute Error', fontsize=18)
plt.title('Error vs 1/√m (Theoretical Rate)', fontsize=20)
plt.legend(fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(ms, errors_raw, 'ro-', linewidth=2, markersize=8, label='Raw Error')
plt.plot(ms, errors_corrected, 'bs-', linewidth=2, markersize=8, label='BGK Corrected Error')
plt.xlabel('Number of Monitoring Steps (m)')
plt.ylabel('Absolute Error')
plt.title('Error Comparison (Linear Scale)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [55]:
class BarrierOptionFD:
    def __init__(self, S0=100, K=100, H=120, T=1, r=0.05, sigma=0.2):

        self.S0 = S0
        self.K = K
        self.H = H
        self.T = T
        self.r = r
        self.sigma = sigma
        
    def price_implicit_fd(self, M=500, N=500):

        S_max = 1.5 * self.H  
        S_min = 0  

        dS = (S_max - S_min) / M
        dt = self.T / N
        S = np.linspace(S_min, S_max, M+1)
        barrier_idx = np.searchsorted(S, self.H)
        V = np.maximum(S - self.K, 0)

        V[barrier_idx:] = 0

        self.S_grid = S
        self.V_history = [V.copy()]

        for n in range(N):
            V_old = V.copy()
            interior_size = min(barrier_idx - 1, M - 1) 
            
            if interior_size <= 0:
                break

            a = np.zeros(interior_size) 
            b = np.zeros(interior_size)    
            c = np.zeros(interior_size)  
            d = np.zeros(interior_size)  
            
            for i in range(1, interior_size + 1):
                Si = S[i]
                alpha = 0.5 * dt * self.sigma**2 * Si**2 / dS**2
                beta = 0.5 * dt * self.r * Si / dS
                #   Building the coeeficient matrix (look back might be wrong tho )
                a[i-1] = -alpha + beta 
                b[i-1] = 1 + 2*alpha + dt*self.r 
                c[i-1] = -alpha - beta 
                d[i-1] = V_old[i]
                
                if i == 1:
                    d[i-1] -= a[i-1] * 0
                    a[i-1] = 0
                    
                if i == interior_size:
                    d[i-1] -= c[i-1] * 0
                    c[i-1] = 0

            if interior_size > 1:
                A = diags([a[1:], b, c[:-1]], [-1, 0, 1], shape=(interior_size, interior_size))
                V_new = spsolve(A, d)
                V[1:interior_size+1] = V_new
            elif interior_size == 1:
                V[1] = d[0] / b[0]

            V[0] = 0  
            V[barrier_idx:] = 0  
            if n % max(1, N//50) == 0:
                self.V_history.append(V.copy())

        price = np.interp(self.S0, S, V)
        
        return price, V, S
    
    def compute_delta(self, M=200, N=500):

        price, V, S = self.price_implicit_fd(M, N)
        delta = np.zeros_like(V)
        dS = S[1] - S[0]
        delta[1:-1] = (V[2:] - V[:-2]) / (2 * dS)
        delta[0] = (V[1] - V[0]) / dS
        delta[-1] = (V[-1] - V[-2]) / dS
        
        return delta, S
    
    def sensitivity_analysis(self):
        """
        Perform sensitivity analysis on key parameters
        """

        base_price, _, _ = self.price_implicit_fd()

        S_range = np.linspace(80, self.H-5, 15) 
        sigma_range = np.linspace(0.1, 0.4, 10)
        r_range = np.linspace(0.01, 0.1, 10)
        T_range = np.linspace(0.5, 2.0, 10)
        H_range = np.linspace(110, 140, 10)

        orig_params = (self.S0, self.sigma, self.r, self.T, self.H)

        prices_S = []
        for S in S_range:
            self.S0 = S
            price, _, _ = self.price_implicit_fd()
            prices_S.append(price)

        self.S0 = orig_params[0]  
        prices_sigma = []
        for sig in sigma_range:
            self.sigma = sig
            price, _, _ = self.price_implicit_fd()
            prices_sigma.append(price)
        
        self.sigma = orig_params[1]  
        prices_r = []
        for r in r_range:
            self.r = r
            price, _, _ = self.price_implicit_fd()
            prices_r.append(price)
        self.r = orig_params[2]  
        prices_T = []
        for T in T_range:
            self.T = T
            price, _, _ = self.price_implicit_fd()
            prices_T.append(price)

        self.T = orig_params[3]  
        prices_H = []
        for H in H_range:
            self.H = H
            price, _, _ = self.price_implicit_fd()
            prices_H.append(price)

        self.S0, self.sigma, self.r, self.T, self.H = orig_params
        
        return {'S': (S_range, prices_S),'sigma': (sigma_range, prices_sigma),'r': (r_range, prices_r),
            'T': (T_range, prices_T),'H': (H_range, prices_H)}
    
    def plot_option_surface(self):

        M, N = 100, 50
        price, V_final, S = self.price_implicit_fd(M, N)
 
        t = np.linspace(0, self.T, len(self.V_history))

        S_mesh, T_mesh = np.meshgrid(S, t)
        V_surface = np.array(self.V_history)

        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')

        surf = ax.plot_surface(S_mesh, T_mesh, V_surface, cmap='viridis', alpha=0.8)

        barrier_line_t = np.linspace(0, self.T, 100)
        barrier_line_S = np.full_like(barrier_line_t, self.H)
        barrier_line_V = np.zeros_like(barrier_line_t)
        ax.plot(barrier_line_S, barrier_line_t, barrier_line_V, 'r-', linewidth=3, label='Barrier')
        ax.set_xlabel('Stock Price (S)')
        ax.set_ylabel('Time to Maturity (T)')
        ax.set_zlabel('Option Value')
        ax.set_title('Barrier Up-and-Out Call Option Surface')
        plt.colorbar(surf)
        plt.legend()
        plt.show()


In [50]:
option = BarrierOptionFD(S0=100, K=90, H=110, T=1, r=0.05, sigma=0.2)

In [ ]:
price, V_final, S = option.price_implicit_fd()
print(f"Option Price: ${price:.6f}")
print(f"Parameters: S0=${option.S0}, K=${option.K}, H=${option.H}, T={option.T}, r={option.r}, σ={option.sigma}")

In [ ]:
delta, S_delta = option.compute_delta()
plt.figure(figsize=(10, 6))
plt.plot(S_delta, delta, 'b-', linewidth=2, label='Delta')
plt.axvline(x=option.H, color='r', linestyle='--', label=f'Barrier (H={option.H})')
plt.axvline(x=option.S0, color='g', linestyle='--', label=f'Current Price (S₀={option.S0})')
plt.xlabel('Stock Price (S)')
plt.ylabel('Delta (∂V/∂S)')
plt.title('Delta of Barrier Up-and-Out Call Option')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(0, 130)
plt.show()

In [ ]:
delta_at_S0 = np.interp(option.S0, S_delta, delta)
print(f"Delta at S0=${option.S0}: {delta_at_S0:.6f}")

In [ ]:
sensitivities = option.sensitivity_analysis()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

colors = ['blue', 'green', 'red', 'orange', 'purple']
titles = ['Stock Price (S)', 'Volatility (σ)', 'Interest Rate (r)', 
            'Time to Maturity (T)', 'Barrier Level (H)']
xlabels = ['Stock Price', 'Volatility', 'Interest Rate', 
            'Time to Maturity', 'Barrier Level']

for i, (param, (x_vals, prices)) in enumerate(sensitivities.items()):
    if i < 5:  
        axes[i].plot(x_vals, prices, color=colors[i], marker='o', linewidth=2, markersize=4)
        axes[i].set_xlabel(xlabels[i])
        axes[i].set_ylabel('Option Price')
        axes[i].set_title(f'Sensitivity to {titles[i]}')
        axes[i].grid(True, alpha=0.3)

fig.delaxes(axes[5])
plt.tight_layout()
plt.show()

In [ ]:
option.plot_option_surface()

In [ ]:
d1 = (np.log(option.S0/option.K) + (option.r + 0.5*option.sigma**2)*option.T) / (option.sigma*np.sqrt(option.T))
d2 = d1 - option.sigma*np.sqrt(option.T)
vanilla_call = option.S0*norm.cdf(d1) - option.K*np.exp(-option.r*option.T)*norm.cdf(d2)
print(f"Vanilla Call Price (upper bound): ${vanilla_call:.6f}")
print(f"Barrier Call Price: {price:.6f}")
print(f"Difference: {vanilla_call - price:.6f}")

